In [6]:
# =================================================================
# 1. SETUP & ENVIRONMENT
# =================================================================
import os
import sys
import json
import logging
import warnings
import gc
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
from joblib import Parallel, delayed

%pip install tensorflow
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

# --- PATH LOGIC (Fixes the "ModuleNotFoundError") ---
# Assuming this notebook is in 'notebooks/' or root
PROJECT_ROOT = Path.cwd() 
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent # Move up if in a subfolder

SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

# --- CONFIGURATION ---
DATA_DIR = PROJECT_ROOT / "data"
BATCH_DIR = DATA_DIR / "processed_batches"
MODELS_DIR = PROJECT_ROOT / "models"
BATCH_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(exist_ok=True)

CONFIG = {
    "sequence_length": 30,
    "n_features": 126,
    "batch_size": 32,
    "epochs": 100,
    "learning_rate": 0.001,
    "save_every_n": 500, # Save checkpoint every 500 videos
}

# Verify Preprocessor
try:
    from hand_sign_detection.training.preprocessor import WlaslPreprocessor
    print("🚀 SUCCESS: Project modules linked correctly.")
except ImportError as e:
    print(f"❌ ERROR: Still can't find modules. Check SRC_PATH: {SRC_PATH}")

Note: you may need to restart the kernel to use updated packages.



A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.4 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "c:\Users\suman\Documents\hand_sign_detection_dynamic\venv\Lib\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "c:\Users\suman\Documents\hand_sign_detection_dynamic\venv\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "c:\Users\suman\Documents\hand_sign_detection_dynamic\venv\Lib\site-packages\ipykernel\kernel

AttributeError: _ARRAY_API not found

❌ ERROR: Still can't find modules. Check SRC_PATH: c:\Users\suman\Documents\hand_sign_detection_dynamic\src


In [2]:
# =================================================================
# 2. ROBUST DATA PROCESSING (BATCHED)
# =================================================================

def process_and_save_batches():
    """Processes 11,998 videos and saves to disk in chunks to save RAM."""
    x_path = DATA_DIR / "X_data.npy"
    y_path = DATA_DIR / "y_data.npy"

    if x_path.exists():
        print("✅ Full dataset found. Skipping processing.")
        return np.load(x_path), np.load(y_path)

    print("🏃 Starting Marathon Processing (12k Videos)...")
    preprocessor = WlaslPreprocessor()
    
    # Load WLASL Metadata
    with open(DATA_DIR / "WLASL_v0.3.json", 'r') as f:
        wlasl_data = json.load(f)

    # Flatten JSON into a list of tasks
    video_tasks = []
    for entry in wlasl_data:
        gloss = entry['gloss']
        for inst in entry['instances']:
            video_id = inst['video_id']
            v_path = DATA_DIR / "videos" / f"{video_id}.mp4"
            if v_path.exists():
                video_tasks.append({"path": str(v_path), "label": gloss})

    print(f"Found {len(video_tasks)} valid videos.")

    X_full, y_full = [], []
    
    # Process in batches to prevent memory leaks
    for i in range(0, len(video_tasks), CONFIG["save_every_n"]):
        batch = video_tasks[i : i + CONFIG["save_every_n"]]
        
        # Parallel Processing of the current batch
        # 'n_jobs=-1' uses all CPU cores
        results = Parallel(n_jobs=-1)(
            delayed(preprocessor.process_video)(item["path"]) for item in batch
        )
        
        # Collect results and keep labels in sync
        for idx, features in enumerate(results):
            if features is not None:
                # Use float32 to save 50% memory
                X_full.append(features.astype('float32'))
                y_full.append(batch[idx]["label"])
        
        print(f"Processed {i + len(batch)} / {len(video_tasks)}")
        gc.collect() # Force clean RAM after every batch

    # Convert to final arrays
    X_final = np.array(X_full)
    y_final = np.array(y_full)
    
    # Save final versions
    np.save(x_path, X_final)
    np.save(y_path, y_final)
    return X_final, y_final

X_raw, y_raw = process_and_save_batches()

✅ Full dataset found. Skipping processing.


In [8]:

# =================================================================
# 3. PREPROCESSING & SPLITTING
# =================================================================

# 1. Clean NaNs
X_raw = np.nan_to_num(X_raw)

# 2. Label Encoding
le = LabelEncoder()
y_int = le.fit_transform(y_raw)
n_classes = len(le.classes_)

# 3. Scale Features (3D arrays need flattening for StandardScaler)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw.reshape(-1, X_raw.shape[-1])).reshape(X_raw.shape)

# 4. Stratified Split (Ensures all classes are represented in Test)
# We filter classes with only 1 sample first
unique, counts = np.unique(y_int, return_counts=True)
valid_classes = unique[counts > 1]
mask = np.isin(y_int, valid_classes)

# Calculate test_size: ensure at least 1 sample per class in test set
n_valid_samples = np.sum(mask)
n_classes_valid = len(valid_classes)
min_test_size = n_classes_valid
test_size = max(min_test_size, int(n_valid_samples * 0.1))
test_size = min(test_size, n_valid_samples - n_classes_valid)  # Leave at least 1 per class in train

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled[mask], 
    tf.keras.utils.to_categorical(y_int[mask], num_classes=n_classes),
    test_size=test_size,
    stratify=y_int[mask],
    random_state=42
)
print(f"✅ Data ready: {X_train.shape[0]} train samples, {X_test.shape[0]} test samples, {n_classes} classes.")

✅ Data ready: 5 train samples, 3 test samples, 6 classes.


In [10]:
# =================================================================
# 4. MODEL & TRAINING
# =================================================================

model = keras.Sequential([
    layers.LSTM(128, return_sequences=True, input_shape=(X_train.shape[1], X_train.shape[2])),
    layers.Dropout(0.3),
    layers.LSTM(64),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(n_classes, activation='softmax')
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Callbacks for long training runs
my_callbacks = [
    callbacks.EarlyStopping(patience=10, restore_best_weights=True),
    callbacks.ReduceLROnPlateau(factor=0.5, patience=5),
    callbacks.CSVLogger(str(MODELS_DIR / "training_log.csv")), # Saves progress if VS Code crashes
    callbacks.ModelCheckpoint(str(MODELS_DIR / "best_model.keras"), save_best_only=True)
]

history = model.fit(
    X_train, y_train,
    validation_split=0.15,
    epochs=CONFIG["epochs"],
    batch_size=CONFIG["batch_size"],
    callbacks=my_callbacks
)

Epoch 1/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step - accuracy: 0.0000e+00 - loss: 1.7852 - val_accuracy: 0.0000e+00 - val_loss: 1.8336 - learning_rate: 0.0010
Epoch 2/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 72ms/step - accuracy: 0.7500 - loss: 1.5350 - val_accuracy: 0.0000e+00 - val_loss: 1.8510 - learning_rate: 0.0010
Epoch 3/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 75ms/step - accuracy: 1.0000 - loss: 1.3731 - val_accuracy: 0.0000e+00 - val_loss: 1.8742 - learning_rate: 0.0010
Epoch 4/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 71ms/step - accuracy: 1.0000 - loss: 1.2805 - val_accuracy: 0.0000e+00 - val_loss: 1.8906 - learning_rate: 0.0010
Epoch 5/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 76ms/step - accuracy: 1.0000 - loss: 1.1503 - val_accuracy: 0.0000e+00 - val_loss: 1.9041 - learning_rate: 0.0010
Epoch 6/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 76ms/step - accuracy: 1.0000 - loss: 0.8829 - val_accuracy: 0.0000e+00 - val_loss: 1.9204 - learning_rate: 0.0010
Epoch 7/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 73ms/step - accuracy: 1.0000 - los

In [11]:
# =================================================================
# 5. EXPORT
# =================================================================
import joblib
joblib.dump(le, MODELS_DIR / "label_encoder.pkl")
joblib.dump(scaler, MODELS_DIR / "scaler.pkl")
model.save(MODELS_DIR / "final_gesture_model.h5")